# Connect-4 Tournament Play — MCTS + Soft Actor-Critic

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Stiles-Clements1/connect4-rl-arena/blob/main/notebooks/tournament_play.ipynb)

Live-match harness for tournament play. Our agent uses Monte-Carlo
tree search guided by the Soft Actor-Critic policy + Q heads (the same
SAC model in `RL models/soft_actor_critic.keras` you've been
evaluating); we just run a configurable number of simulations per move
and take the most-visited child of the root, so the agent can see
multi-ply tactics and double-threat positions that the raw policy
misses.

**How to use during tournament:**
1. Run Cell 1 (loads the model, builds the MCTS agent, **warms up
   `@tf.function`** so the first real move isn't slow).
2. Run Cell 2 (defines the clickable game UI).
3. Run Cell 3 to display the board. Click **🤖 Run MCTS** when it's
   our turn, then click the highlighted column to confirm. Click any
   column on the opponent's turn to record their move. Use **↩ Undo**
   if you mis-click. **🔄 New game** to reset between games.

**Strength vs. time trade-off** is controlled by `N_SIMULATIONS` in
Cell 1. Higher = stronger but slower. Rough rule of thumb on Colab T4
GPU at the default `N_PARALLEL_SIMS=16`:

| N_SIMULATIONS | est. time/move |
|---|---|
| 200 | ~0.4 s |
| 400 | ~0.8 s |
| 800 | ~1.5 s |
| **1000 (default)** | **~1.9 s** |
| 1600 | ~3.0 s |

Tune to fit whatever per-move time budget the tournament gives you.

**Standard Connect-4 rules apply.** The agent uses `ge.legal_moves(board)`
internally so it cannot pick a full column under any circumstance —
PUCT only sees children for legal moves, and the column buttons in the
UI disable themselves once a column fills up. The same applies if you
override the agent's suggestion: full columns are unclickable. Undo
peels back the last played piece (either side) so a mistaken click is
recoverable in one button.


In [1]:
# Cell 1 — setup, load SAC, build MCTSAgent.

# ── CONFIG ──────────────────────────────────────────────────────────────────
# Strength / speed knobs. Tune these for your tournament time budget.
N_SIMULATIONS    = 6000   # MCTS rollouts per agent move (higher = stronger, slower)
N_PARALLEL_SIMS  = 16     # leaves evaluated per batched forward pass (8-32)
C_PUCT           = 1.4    # exploration constant; 1.0-2.0 range is standard
VALUE_METHOD     = 'mean_q'  # 'mean_q' (uses SAC Q heads) or 'rollout' (random playouts)
USE_TACTICS      = True   # immediate-win / immediate-block before MCTS

# MCTS internal exploration. Dirichlet noise is mixed into the root
# prior so different simulations see slightly different priors and
# diverge to different child subtrees instead of all stacking onto the
# network's top pick. This is the standard AlphaZero exploration
# technique. The AGENT still plays deterministic argmax over the final
# visit counts, so our actual move is whatever MCTS judges strongest.
# This noise is INSIDE the search only -- it does not weaken our play.
ADD_ROOT_NOISE   = True   # Dirichlet(0.3) noise on root prior at weight 0.25

# Display knobs.
OUR_NAME         = 'Group 12'
OPP_NAME         = 'Opponent'
WE_GO_FIRST      = True   # default for play(); override per-game with play(we_go_first=False)
# ────────────────────────────────────────────────────────────────────────────

import os, sys, subprocess, time
from pathlib import Path

GITHUB_OWNER  = 'Stiles-Clements1'
GITHUB_REPO   = 'connect4-rl-arena'
GITHUB_BRANCH = 'main'
GITHUB_URL    = f'https://github.com/{GITHUB_OWNER}/{GITHUB_REPO}.git'

IS_COLAB = 'google.colab' in sys.modules

if IS_COLAB:
    REPO_ROOT = Path(f'/content/{GITHUB_REPO}')
    if not REPO_ROOT.exists():
        print(f'Cloning {GITHUB_URL} (branch {GITHUB_BRANCH}) → {REPO_ROOT}')
        subprocess.run(
            ['git', 'clone', '--branch', GITHUB_BRANCH, GITHUB_URL, str(REPO_ROOT)],
            check=True,
        )
    else:
        print(f'Repo already at {REPO_ROOT}; pulling latest {GITHUB_BRANCH}.')
        subprocess.run(['git', '-C', str(REPO_ROOT), 'fetch',  '--quiet', 'origin'], check=False)
        subprocess.run(['git', '-C', str(REPO_ROOT), 'checkout','--quiet', GITHUB_BRANCH], check=False)
        subprocess.run(['git', '-C', str(REPO_ROOT), 'pull',   '--quiet', 'origin', GITHUB_BRANCH], check=False)
else:
    # Local: walk up from the notebook's CWD looking for the repo root.
    # If we don't find it (e.g. the notebook was downloaded standalone for
    # grading), clone the repo into ./connect4-rl-arena so this notebook
    # is fully self-bootstrapping with just internet + git + python.
    here = Path.cwd().resolve()
    REPO_ROOT = next(
        (p for p in [here, *here.parents] if (p / 'src' / 'game_engine.py').exists()),
        None,
    )
    if REPO_ROOT is None:
        REPO_ROOT = here / GITHUB_REPO
        if not REPO_ROOT.exists():
            print(f'No repo found in directory tree from {here}.')
            print(f'Cloning {GITHUB_URL} (branch {GITHUB_BRANCH}) → {REPO_ROOT}...')
            subprocess.run(
                ['git', 'clone', '--branch', GITHUB_BRANCH, GITHUB_URL, str(REPO_ROOT)],
                check=True,
            )
        else:
            print(f'Repo already at {REPO_ROOT}; pulling latest {GITHUB_BRANCH}.')
            subprocess.run(['git', '-C', str(REPO_ROOT), 'fetch',  '--quiet', 'origin'], check=False)
            subprocess.run(['git', '-C', str(REPO_ROOT), 'checkout','--quiet', GITHUB_BRANCH], check=False)
            subprocess.run(['git', '-C', str(REPO_ROOT), 'pull',   '--quiet', 'origin', GITHUB_BRANCH], check=False)
    else:
        print(f'Local repo root: {REPO_ROOT}')

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)

# Force-reimport src.* so a freshly-pulled session picks up new code
for _m in [k for k in sys.modules if k == 'src' or k.startswith('src.')]:
    del sys.modules[_m]

# Quiet TF down before importing it
os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '2')
import tensorflow as tf
tf.get_logger().setLevel('ERROR')
_gpus = tf.config.list_physical_devices('GPU')
for _g in _gpus:
    try:
        tf.config.experimental.set_memory_growth(_g, True)
    except Exception:
        pass
HARDWARE = f'GPU ({_gpus[0].name.split(":")[-1]})' if _gpus else 'CPU only'
print(f'Hardware = {HARDWARE}')
print(f'Mode     = {"Colab" if IS_COLAB else "local"}')
print(f'REPO_ROOT = {REPO_ROOT}')

import numpy as np
from src import model_loader as ml
from src import game_engine as ge
from src.mcts import MCTSAgent

# Load the SAC model. Auto-detect encoding from input shape:
#   (None, 6, 7, 2) -> 'B'   (None, 6, 7, 1) -> 'A'   (None, 42, 2) -> 'B_flat'
SAC_PATH = REPO_ROOT / 'RL models' / 'soft_actor_critic.keras'
keras_model = tf.keras.models.load_model(str(SAC_PATH), compile=False)
try:
    in_shape = tuple(keras_model.input.shape)
except (AttributeError, TypeError):
    in_shape = tuple(keras_model.inputs[0].shape)
trailing = in_shape[-3:] if len(in_shape) >= 3 else None
if trailing == (6, 7, 2):
    encoding = 'B'
elif trailing == (6, 7, 1):
    encoding = 'A'
elif in_shape[-2:] == (42, 2):
    encoding = 'B_flat'
else:
    raise ValueError(f'Unrecognised input shape {in_shape}')
sac_wrapper = ml.ModelWrapper(keras_model, encoding, name='SAC')
print(f'\nLoaded SAC: input shape {in_shape}, encoding={encoding}')

# Build the MCTS agent
mcts_agent = MCTSAgent(
    sac_wrapper,
    n_simulations=N_SIMULATIONS,
    c_puct=C_PUCT,
    value_method=VALUE_METHOD,
    n_parallel_sims=N_PARALLEL_SIMS,
    use_tactics=USE_TACTICS,
    add_root_noise=ADD_ROOT_NOISE,
)

print(f'\nAgent: {mcts_agent.name}')
print(f'  N_SIMULATIONS    = {N_SIMULATIONS}')
print(f'  N_PARALLEL_SIMS  = {N_PARALLEL_SIMS}')
print(f'  C_PUCT           = {C_PUCT}')
print(f'  VALUE_METHOD     = {VALUE_METHOD}')
print(f'  USE_TACTICS      = {USE_TACTICS}')

# Warm-up move. The first @tf.function call triggers a (slow) graph
# trace; doing it now means the first move of the actual game isn't
# the one that pays the trace cost.
print(f'\nWarming up @tf.function (first call traces; this takes a few seconds)...')
_warm_t0 = time.perf_counter()
_ = mcts_agent.select_move(np.zeros((6, 7), dtype=np.int8), 1)
print(f'  warm-up move took {time.perf_counter() - _warm_t0:.1f}s')

# A second move to measure post-warm-up speed (this is roughly what
# real moves will cost during the game).
_warm_t0 = time.perf_counter()
_ = mcts_agent.select_move(np.zeros((6, 7), dtype=np.int8), 1)
_post_warm_ms = (time.perf_counter() - _warm_t0) * 1000
print(f'  steady-state move took {_post_warm_ms:.0f}ms')
print(f'\nReady to play.')


Cloning https://github.com/Stiles-Clements1/connect4-rl-arena.git (branch main) → /content/connect4-rl-arena
Hardware = GPU (0)
Mode     = Colab
REPO_ROOT = /content/connect4-rl-arena

Loaded SAC: input shape (None, 6, 7, 2), encoding=B

Agent: mcts_n6000_mean_q
  N_SIMULATIONS    = 6000
  N_PARALLEL_SIMS  = 16
  C_PUCT           = 1.4
  VALUE_METHOD     = mean_q
  USE_TACTICS      = True

Warming up @tf.function (first call traces; this takes a few seconds)...
  warm-up move took 3.4s
  steady-state move took 2615ms

Ready to play.


In [2]:
# Cell 2 — clickable widget UI for tournament play.
#
# Same logic as the agent itself; just a different input/display layer.
# Single board, click columns to drop pieces, click 'Run MCTS' to
# compute our move (which highlights the suggested column - click the
# column to confirm, or click a different one to override). 'Undo'
# peels back the last move, 'New game' resets.

import ipywidgets as widgets
from IPython.display import display

class GameUI:
    """Widget-based Connect-4 UI for tournament play.

    - Status line shows whose turn it is and any game-over message.
    - The board renders below as 🔴 / 🟡 / ⚪ glyphs.
    - 7 column buttons drop a piece in that column for the current player.
    - 'Run MCTS' computes our agent's preferred column (highlighted in
      green; per-move wallclock printed). Click that column to confirm,
      or click a different column to override.
    - 'Undo' takes back the last move (any side).
    - 'New game (we first / opp first)' resets the board.
    """

    def __init__(self, agent, we_go_first: bool = True):
        self.agent = agent
        self.history: list[tuple[int, int]] = []   # (col, player) for undo
        self.board = np.zeros((6, 7), dtype=np.int8)
        self.current_player = 1   # red always moves first by Connect-4 rules
        self._set_we_go_first(we_go_first)

        # ── Widgets ───────────────────────────────────────────────────────
        self.status = widgets.HTML()
        self.board_view = widgets.HTML()
        self.suggestion = widgets.HTML()

        # 7 column buttons
        self.col_buttons = [
            widgets.Button(
                description=str(i + 1),
                layout=widgets.Layout(width='52px', height='38px'),
            )
            for i in range(7)
        ]
        for i, btn in enumerate(self.col_buttons):
            btn.on_click(lambda _b, c=i: self._on_col_click(c))

        self.compute_btn = widgets.Button(
            description='🤖 Run MCTS',
            button_style='primary',
            tooltip='Compute our agent\'s move with MCTS+SAC',
            layout=widgets.Layout(width='160px'),
        )
        self.compute_btn.on_click(self._on_compute)

        self.undo_btn = widgets.Button(
            description='↩ Undo',
            button_style='warning',
            tooltip='Peel back the last move (either side)',
            layout=widgets.Layout(width='100px'),
        )
        self.undo_btn.on_click(self._on_undo)

        self.reset_we_btn = widgets.Button(
            description='🔄 New (we first)',
            button_style='info',
            layout=widgets.Layout(width='150px'),
        )
        self.reset_we_btn.on_click(lambda _b: self._on_reset(True))

        self.reset_opp_btn = widgets.Button(
            description='🔄 New (opp first)',
            button_style='info',
            layout=widgets.Layout(width='150px'),
        )
        self.reset_opp_btn.on_click(lambda _b: self._on_reset(False))

        # ── Layout ───────────────────────────────────────────────────────
        self.box = widgets.VBox([
            self.status,
            self.board_view,
            widgets.HBox(self.col_buttons),
            widgets.HBox([self.compute_btn, self.suggestion]),
            widgets.HBox([self.undo_btn, self.reset_we_btn, self.reset_opp_btn]),
        ])

        self._refresh()

    # ── public ────────────────────────────────────────────────────────────
    def display(self):
        display(self.box)

    # ── internals ─────────────────────────────────────────────────────────
    def _set_we_go_first(self, we_go_first: bool):
        self.we_go_first = we_go_first
        self.our_side = 1 if we_go_first else -1

    def _board_html(self) -> str:
        # Render as an HTML table so columns align exactly. <pre> + monospace
        # doesn't work here because emoji glyphs occupy a different visual
        # width than ASCII digits, so the column numbers drift to the right
        # of where the actual columns sit.
        cell_w = 44  # pixels — wide enough for the emoji at 28px
        header = '<tr>' + ''.join(
            f'<td style="width:{cell_w}px; text-align:center; '
            f'color:#888; font-size:14px; padding:4px 0;">{i+1}</td>'
            for i in range(7)
        ) + '</tr>'
        body_rows = []
        for r in range(6):
            cells = []
            for c in range(7):
                v = self.board[r, c]
                emoji = '🔴' if v == 1 else '🟡' if v == -1 else '⚪'
                cells.append(
                    f'<td style="width:{cell_w}px; height:{cell_w}px; '
                    f'text-align:center; font-size:28px; padding:0;">{emoji}</td>'
                )
            body_rows.append('<tr>' + ''.join(cells) + '</tr>')
        return (
            '<table style="border-collapse:collapse; margin:8px 0; '
            'background:#222; border-radius:6px; padding:6px;">'
            + header
            + ''.join(body_rows)
            + '</table>'
        )

    def _refresh(self):
        our_token = '🔴' if self.our_side == 1 else '🟡'
        opp_token = '🟡' if self.our_side == 1 else '🔴'

        # Mode header: who's going first in this game. Always visible so
        # there's never confusion mid-game about which side is which.
        if self.we_go_first:
            mode = (
                f'<div style="background:#eafaf1; padding:8px 12px; '
                f'border-left:4px solid #2ecc71; border-radius:4px; '
                f'margin-bottom:6px;">'
                f'<b>This game: WE go first</b> — '
                f'we are {our_token} {OUR_NAME}, opponent is {opp_token} {OPP_NAME}'
                f'</div>'
            )
        else:
            mode = (
                f'<div style="background:#fdf2e9; padding:8px 12px; '
                f'border-left:4px solid #e67e22; border-radius:4px; '
                f'margin-bottom:6px;">'
                f'<b>This game: OPPONENT goes first</b> — '
                f'we are {our_token} {OUR_NAME}, opponent is {opp_token} {OPP_NAME}'
                f'</div>'
            )

        done, winner = ge.is_terminal(self.board)
        if done:
            if winner == 0:
                turn = '<h3 style="color:#777; margin:4px 0;">Draw!</h3>'
            elif winner == self.our_side:
                turn = f'<h3 style="color:#2ecc71; margin:4px 0;">{our_token} {OUR_NAME} wins! 🎉</h3>'
            else:
                turn = f'<h3 style="color:#e74c3c; margin:4px 0;">{opp_token} {OPP_NAME} wins.</h3>'
        elif self.current_player == self.our_side:
            turn = (
                f'<b>{our_token} {OUR_NAME}\'s turn</b> — click '
                f'<b>🤖 Run MCTS</b> to compute, or click a column to override.'
            )
        else:
            turn = (
                f'<b>{opp_token} {OPP_NAME}\'s turn</b> — '
                f'click the column they just played.'
            )
        self.status.value = mode + turn
        self.board_view.value = self._board_html()

        # Enable / disable column buttons (full columns greyed out, all
        # disabled when the game is over) and clear any green highlight
        # from the previous Run-MCTS suggestion.
        # NOTE: ipywidgets v8+ requires Python `bool`, not numpy `bool_`,
        # so wrap each assignment in bool() — board[0, i] != 0 is np.bool_,
        # and `done` (from ge.is_terminal) can also be np.bool_.
        for i, btn in enumerate(self.col_buttons):
            btn.disabled = bool(done or self.board[0, i] != 0)
            btn.button_style = ''
        self.compute_btn.disabled = bool(done or self.current_player != self.our_side)
        self.undo_btn.disabled = bool(len(self.history) == 0)

    def _on_col_click(self, col: int):
        if self.board[0, col] != 0:
            return
        self.board = ge.make_move(self.board, col, self.current_player)
        self.history.append((col, self.current_player))
        self.current_player = -self.current_player
        self.suggestion.value = ''
        self._refresh()

    def _on_compute(self, _btn):
        if self.current_player != self.our_side:
            return
        # Disable the compute button while MCTS runs (gives a small visual
        # hint that the click registered, even though ipywidgets handlers
        # are synchronous and the UI won't update until MCTS returns).
        self.compute_btn.disabled = bool(True)
        self.suggestion.value = '<i>Running MCTS...</i>'
        try:
            t0 = time.perf_counter()
            col = int(self.agent.select_move(self.board, self.current_player))
            elapsed_ms = (time.perf_counter() - t0) * 1000
        finally:
            self.compute_btn.disabled = bool(False)
        # Highlight the suggested column
        for i, b in enumerate(self.col_buttons):
            b.button_style = 'success' if i == col else ''
        self.suggestion.value = (
            f'<b>→ Click column {col + 1}</b>  '
            f'<span style="color:#666">({elapsed_ms:.0f} ms, '
            f'N_SIMULATIONS={N_SIMULATIONS})</span>'
        )

    def _on_undo(self, _btn):
        if not self.history:
            return
        col, player = self.history.pop()
        # Topmost piece of `player` in column `col` is the lowest row index
        # whose value matches; remove it.
        for r in range(6):
            if self.board[r, col] == player:
                self.board[r, col] = 0
                break
        self.current_player = -self.current_player
        self.suggestion.value = ''
        self._refresh()

    def _on_reset(self, we_go_first: bool):
        self.board = np.zeros((6, 7), dtype=np.int8)
        self.history = []
        self.current_player = 1
        self._set_we_go_first(we_go_first)
        self.suggestion.value = ''
        self._refresh()

print('GameUI class defined. Run the next cell to start a game.')


GameUI class defined. Run the next cell to start a game.


In [3]:
# Cell 3 — instantiate and display the game UI.
#
# Run this cell to start a tournament game. Use the buttons:
# - 🤖 Run MCTS  (when it's your turn) → computes our move; click the
#                 highlighted column to confirm.
# - 1-7         → drop a piece in that column for the current player.
# - ↩ Undo      → peel back the last move (any side; press as many
#                 times as needed if you mis-clicked a streak).
# - 🔄 New game → reset the board.

game = GameUI(mcts_agent, we_go_first=WE_GO_FIRST)
game.display()
